# Calcolo Embedding E5-base — Google Colab (GPU)

**Prima di eseguire**: Runtime → Cambia tipo di runtime → **T4 GPU**

### Flusso
1. Installa `sentence-transformers`
2. Carica `dataset_clean.csv` (generato dal notebook Preprocessing locale)
3. Split temporale identico al classifier locale (`< 2025-11-01` / `>= 2025-11-01`)
4. Calcola embedding `intfloat/multilingual-e5-base` su GPU
5. Scarica `priority_e5_train.npy` e `priority_e5_test.npy`
   → metti questi file in `TicketClassifier/embeddings/` sul tuo PC

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import files

print("Carica dataset_clean.csv (generato da Preprocessing.ipynb)")
uploaded = files.upload()

# Verifica che il file sia stato caricato
import os
assert 'dataset_clean.csv' in uploaded, "File non trovato: assicurati di caricare 'dataset_clean.csv'"
print(f"OK — {len(uploaded['dataset_clean.csv']):,} bytes")

In [ ]:
import torch
import pandas as pd
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM disponibile: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ATTENZIONE: nessuna GPU rilevata. Vai su Runtime → Cambia tipo di runtime → T4 GPU")

In [ ]:

SOGLIA_SPLIT = '2025-11-01'

# engine='python': più lento del C-parser ma gestisce correttamente i campi
# con newline embedded (es. descrizioni multiriga nei ticket).
# Necessario finché non si rigenera dataset_clean.csv dal Preprocessing aggiornato.
df = pd.read_csv(
    'dataset_clean.csv',
    parse_dates=['data_creazione'],
    engine='python',
)

df_train = df[df['data_creazione'] < SOGLIA_SPLIT].copy()
df_test  = df[df['data_creazione'] >= SOGLIA_SPLIT].copy()

print(f"Dataset totale : {len(df):,} ticket")
print(f"Train (<{SOGLIA_SPLIT}) : {len(df_train):,}")
print(f"Test  (>={SOGLIA_SPLIT}): {len(df_test):,}")
print()
print("Distribuzione priorità — train:")
print(df_train['priorita_finale'].value_counts().sort_index())
print()
print("Distribuzione priorità — test:")
print(df_test['priorita_finale'].value_counts().sort_index())


In [ ]:
from sentence_transformers import SentenceTransformer
import time

MODEL_NAME = 'intfloat/multilingual-e5-base'

print(f"Carico modello {MODEL_NAME} su {device}...")
model = SentenceTransformer(MODEL_NAME, device=device)
print("Modello caricato.")
print()

# Il prefisso 'query: ' è richiesto dal modello E5 per testi di query/input.
# batch_size=256 funziona bene su T4 (16 GB); abbassa a 128 se vedi OOM.

print("[1/2] Calcolo embedding TRAIN...")
t0 = time.time()
emb_train = model.encode(
    ("query: " + df_train['testo_input'].astype(str)).tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f"TRAIN completato in {time.time()-t0:.0f}s — shape: {emb_train.shape}")
print()

print("[2/2] Calcolo embedding TEST...")
t0 = time.time()
emb_test = model.encode(
    ("query: " + df_test['testo_input'].astype(str)).tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f"TEST completato in {time.time()-t0:.0f}s — shape: {emb_test.shape}")

In [ ]:
np.save('priority_e5_train.npy', emb_train)
np.save('priority_e5_test.npy',  emb_test)

# Verifica integrità (ricarica e confronta)
assert np.allclose(np.load('priority_e5_train.npy'), emb_train), "Errore: file train corrotto"
assert np.allclose(np.load('priority_e5_test.npy'),  emb_test),  "Errore: file test corrotto"

train_mb = os.path.getsize('priority_e5_train.npy') / 1e6
test_mb  = os.path.getsize('priority_e5_test.npy')  / 1e6

print(f"priority_e5_train.npy — {emb_train.shape} — {train_mb:.0f} MB")
print(f"priority_e5_test.npy  — {emb_test.shape}  — {test_mb:.0f} MB")
print()
print("Tutto OK. Esegui la cella successiva per scaricare i file.")

In [ ]:
from google.colab import files

print("Download priority_e5_train.npy...")
files.download('priority_e5_train.npy')

print("Download priority_e5_test.npy...")
files.download('priority_e5_test.npy')

print()
print("Copia entrambi i file in:")
print("  TicketClassifier/embeddings/priority_e5_train.npy")
print("  TicketClassifier/embeddings/priority_e5_test.npy")
print()
print("Poi ri-esegui PriorityClassifier_v2_VSCode.ipynb da STEP 1.")